# Phase 3 — BoT-SORT from retained Phase 2 detections

**REQUIREMENT:** Run all cells in a fresh Kaggle session. Attach the traffic video dataset and the retained Phase 2 output/cache. This notebook never loads YOLO weights or runs detection. Change the explicit input variables below only when an actually retained, reviewed parent run is available.

**PROPOSAL — controlled, untuned baseline:** Ultralytics 8.4.175 BoT-SORT, published `botsort.yaml` defaults, with ReID disabled until Phase 4 and GMC disabled (`none`) pending a separate decision. All raw model classes remain available. These settings do not remove ReID from the target architecture or settle suitable production thresholds.

Association/Kalman operations run on CPU. CUDA inventory is recorded separately; GPU availability does not mean this cached tracking run executes GPU inference. No geometry, direction, crossing, vehicle totals, stabilization, enhancement or multi-GPU processing is implemented.

Outputs are separate immutable run directories. `tracking_report.json` preserves partial failure status. Temporal bursts inherit the full preceding tracker history. Subsequent visual review must use actual saved images/clips and records; the source notebook alone supplies no findings.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import json, os, platform, subprocess, sys, time, uuid

VIDEO_PATH = Path('/kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-PAGI.mp4')
DETECTION_REPORT_PATH = None  # Optional explicit Path; otherwise discover one exact parent run.
FRAME_DETECTIONS_PATH = None  # Optional explicit Path; otherwise resolve the parent report artifact.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/botsort_tracking')
PARENT_RUN_ID = '20261011T052950_360560Z_ed399fb8'
EXPECTED_VIDEO_SHA256 = '606cfdd26c2d0a6892f5680d6a35d7e4d372efd76dd6bc56e2e4cbed7a74754c'
EXPECTED_CACHE_SHA256 = '033a2815d3abbd23327936a1b9d948b2b9b7d312796516e5ee46aec5685be345'
EXPECTED_FRAME_COUNT = 10282
PACKAGE_VERSION = '8.4.175'
LAP_VERSION = '0.5.13'
TRACKER_OVERRIDES = {'with_reid': False, 'gmc_method': 'none'}
BURSTS = [(0, 29), (2555, 2584), (5125, 5154), (7696, 7725), (10252, 10281)]
REPRESENTATIVE_INDICES = [0, 2570, 5140, 7711, 10281]
FOCUS_INDICES = list(range(7711, 7719))
FOCUS_CROP_XYXY = (500, 250, 1280, 720)  # Display-only audit crop; never used for association/eligibility.
SEED = 0

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ_') + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'samples').mkdir()
(RUN_DIR / 'temporal').mkdir()
os.environ['YOLO_CONFIG_DIR'] = str(RUN_DIR / '.ultralytics_settings')
Path(os.environ['YOLO_CONFIG_DIR']).mkdir()
REPORT = {
    'phase': 3, 'run_id': RUN_ID, 'started_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'initialized', 'claim_type': 'EXPERIMENT_RESULT_ON_EXECUTION',
    'scope': 'one video; cached detections to CPU BoT-SORT; no detector inference or traffic counts',
    'configuration': {'video_path': str(VIDEO_PATH), 'parent_run_id': PARENT_RUN_ID,
                      'tracker_overrides': TRACKER_OVERRIDES, 'seed': SEED,
                      'baseline_status': 'PROPOSAL: published, untuned tracker defaults with explicit isolation overrides'},
    'stages': {}, 'frames_processed': 0,
    'manual_review_status': 'pending_actual_temporal_evidence_review',
}

def write_report():
    (RUN_DIR / 'tracking_report.json').write_text(json.dumps(REPORT, indent=2, allow_nan=False)+'\n')
    lines = ['# Phase 3 execution report', '', f"Status: `{REPORT['status']}`",
             f"Run: `{RUN_ID}`", f"Parent: `{PARENT_RUN_ID}`", '',
             'Cached BoT-SORT association only; no Phase 2 rerun, geometry, counting or ReID experiment.',
             f"Frames processed: {REPORT['frames_processed']}",
             'Unique tracker IDs are lifecycle diagnostics, not physical vehicle or traffic totals.',
             f"Visual review: {REPORT['manual_review_status']}"]
    if 'failure' in REPORT: lines.extend(['', 'Failure: '+REPORT['failure']['message']])
    if 'diagnostics' in REPORT:
        lines.extend(['', '## Observed software/lifecycle diagnostics', '',
                      '```json', json.dumps(REPORT['diagnostics'], indent=2), '```'])
    (RUN_DIR / 'tracking_report.md').write_text('\n'.join(lines)+'\n')

from contextlib import contextmanager
@contextmanager
def stage(name):
    REPORT['current_stage'] = name
    started = time.perf_counter()
    write_report()
    try:
        yield
    except Exception as exc:
        REPORT['status'] = 'failed_partial_execution'
        REPORT['failure'] = {'stage': name, 'type': type(exc).__name__, 'message': str(exc)}
        raise
    finally:
        REPORT['stages'][name] = {'wall_seconds': time.perf_counter()-started}
        write_report()

write_report()
print('Run directory:', RUN_DIR)


In [ ]:
with stage('dependency_installation'):
    wanted = {'ultralytics': PACKAGE_VERSION, 'lap': LAP_VERSION}
    missing = []
    for package, version in wanted.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            if package in sys.modules:
                raise RuntimeError(f'{package} already imported with a different version; restart this session.')
            missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *missing], check=True)
    for package, version in wanted.items():
        assert metadata.version(package) == version, (package, metadata.version(package))


In [ ]:
import hashlib, math, statistics, random
from collections import Counter
import cv2
import numpy as np
import torch
import ultralytics
from ultralytics.engine.results import Boxes
from ultralytics.trackers.bot_sort import BOTSORT
from ultralytics.trackers.basetrack import TrackState
from ultralytics.utils import YAML, IterableSimpleNamespace
from PIL import Image, ImageDraw

random.seed(SEED)
np.random.seed(SEED)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

with stage('environment_and_verified_tracker_source'):
    package_root = Path(ultralytics.__file__).resolve().parent
    EXPECTED_SOURCE_HASHES = {'trackers/bot_sort.py': 'b157cbc33f75f1132df530f04361824bc39a30d4051ce6df1cd5c388e49b3117', 'trackers/byte_tracker.py': 'fb45d70e952f99984085788bc09fc0c60a18b14bc4329b28e32bc2c70ac37401', 'trackers/basetrack.py': '9ef094b9a54ff2f52c9118b5bd1a2f84ba331aa557277f54186de11728c2f7e8', 'trackers/utils/matching.py': '318bf81030558863a4ec6effea3c9d063a297f299367e379a58b89c24e3f421f', 'trackers/utils/gmc.py': 'd797f701a22347001d780b0dc4d0f2f5cabdfefb895c36bbab084ceb2dfc4000', 'trackers/utils/stracks.py': '9534294af850c3a6755c3a5a1ba9b2b068c5b79a11166c8547652944e2632f58', 'trackers/utils/reid.py': '75c54a4f8fe9b1e6946d61b530fbb790fdfcc27df120fa71cc7759c11e9f0475', 'cfg/trackers/botsort.yaml': '8cd88930d3be5557be0777f384eecff0a36a7011850fe613fc0a68178bfd5b01', 'engine/results.py': '871e5b3eed1558c3c048d99058ddf28a7927c1317d020981e7c0836f18e00ee3'}
    source_checks = []
    for relative_path, expected_hash in EXPECTED_SOURCE_HASHES.items():
        path = package_root / relative_path
        actual_hash = sha256_file(path)
        require(actual_hash == expected_hash, f'Pinned tracker source mismatch: {relative_path}')
        source_checks.append({'path': relative_path, 'sha256': actual_hash, 'verified': True})
    cuda_devices = []
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            device = torch.cuda.get_device_properties(index)
            cuda_devices.append({'index': index, 'name': device.name, 'total_memory_bytes': device.total_memory})
    try:
        smi = subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                              '--format=csv,noheader'], capture_output=True, text=True, timeout=15)
        smi_observation = {'returncode': smi.returncode, 'stdout': smi.stdout.strip(), 'stderr': smi.stderr.strip()}
    except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
        smi_observation = {'unavailable': type(exc).__name__}
    REPORT['environment'] = {
        'python': sys.version, 'platform': platform.platform(), 'cpu_count': os.cpu_count(),
        'package_versions': {name: metadata.version(name) for name in
                             ['ultralytics', 'lap', 'torch', 'numpy', 'Pillow']},
        'opencv_version': cv2.__version__, 'cuda_available': torch.cuda.is_available(),
        'visible_cuda_device_count': len(cuda_devices), 'cuda_devices': cuda_devices,
        'nvidia_smi': smi_observation, 'association_device': 'cpu',
        'detector_inference_executed': False, 'reid_inference_executed': False,
        'exact_cross_runtime_repeatability_claimed': False,
    }
    REPORT['tracker_source_verification'] = source_checks
    print(json.dumps(REPORT['environment'], indent=2))


In [ ]:
def discover_parent_report(explicit_path, search_root, expected_run_id):
    if explicit_path is not None:
        path = Path(explicit_path)
        data = json.loads(path.read_text())
        require(data.get('run_id') == expected_run_id, 'Explicit parent report has the wrong run_id.')
        return path, data
    matches = []
    for path in Path(search_root).rglob('detection_report.json'):
        try:
            data = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if data.get('run_id') == expected_run_id:
            matches.append((path, data))
    require(len(matches) == 1,
            f'Expected one mounted parent report for {expected_run_id}; found {len(matches)}. '
            'Attach retained Phase 2 outputs/cache, or set DETECTION_REPORT_PATH explicitly. No YOLO rerun is allowed.')
    return matches[0]

def verify_input_hash(path, expected):
    require(Path(path).is_file(), f'Input file not found: {path}')
    actual = sha256_file(path)
    require(actual == expected, f'SHA-256 mismatch for {path}: {actual}')
    return actual

def validate_detection_row(row, expected_index, video_sha, width, height, class_names):
    require(type(row.get('frame_index')) is int and row['frame_index'] == expected_index,
            f'Out-of-order/missing/extra cached frame: expected {expected_index}.')
    require(row.get('video_sha256') == video_sha, f'Wrong video identity at frame {expected_index}.')
    require(row.get('width') == width and row.get('height') == height,
            f'Wrong dimensions at frame {expected_index}.')
    require(isinstance(row.get('detections'), list), 'Detections must be a list.')
    for index, det in enumerate(row['detections']):
        cls = det.get('class_id')
        require(type(cls) is int and str(cls) in class_names, f'Invalid raw class at {expected_index}/{index}.')
        require(det.get('class_name') == class_names[str(cls)], 'Class name does not match the retained model mapping.')
        confidence = det.get('confidence')
        require(isinstance(confidence, (int, float)) and math.isfinite(confidence) and 0 <= confidence <= 1,
                'Invalid detection confidence.')
        box = det.get('bbox_xyxy_pixels')
        require(isinstance(box, list) and len(box) == 4 and all(isinstance(v, (int, float)) and math.isfinite(v) for v in box),
                'Invalid raw bounding box.')
        x1, y1, x2, y2 = box
        require(0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height, 'Raw detection box must be positive and in-frame.')
    return row

with stage('retained_input_identity_and_cache_validation'):
    DETECTION_REPORT_PATH, PARENT_REPORT = discover_parent_report(DETECTION_REPORT_PATH, INPUT_ROOT, PARENT_RUN_ID)
    require(PARENT_REPORT.get('phase') == 2, 'Parent must be an actual Phase 2 report.')
    require(PARENT_REPORT['input']['sha256'] == EXPECTED_VIDEO_SHA256, 'Parent video hash does not match selected input.')
    require(PARENT_REPORT['artifacts']['frame_detections_sha256'] == EXPECTED_CACHE_SHA256, 'Parent cache hash does not match.')
    require(PARENT_REPORT['execution']['frames_processed'] == EXPECTED_FRAME_COUNT, 'Parent frame total differs from scope.')
    FRAME_DETECTIONS_PATH = (Path(FRAME_DETECTIONS_PATH) if FRAME_DETECTIONS_PATH is not None
                             else DETECTION_REPORT_PATH.parent / PARENT_REPORT['artifacts']['frame_detections'])
    require(FRAME_DETECTIONS_PATH.suffix == '.jsonl', 'Use the retained uncompressed JSONL (not a silently substituted cache).')
    video_hash = verify_input_hash(VIDEO_PATH, EXPECTED_VIDEO_SHA256)
    cache_hash = verify_input_hash(FRAME_DETECTIONS_PATH, EXPECTED_CACHE_SHA256)
    CLASS_NAMES = PARENT_REPORT['model']['class_names']
    WIDTH = int(PARENT_REPORT['input']['opencv_metadata']['width'])
    HEIGHT = int(PARENT_REPORT['input']['opencv_metadata']['height'])
    FPS = float(PARENT_REPORT['input']['opencv_metadata']['fps'])
    require(math.isfinite(FPS) and FPS > 0, 'Parent FPS unavailable; buffer semantics cannot assume 30 FPS.')
    CACHE_ROWS = []
    with FRAME_DETECTIONS_PATH.open() as handle:
        for index, line in enumerate(handle):
            require(index < EXPECTED_FRAME_COUNT, 'Unexpected extra detection row.')
            row = json.loads(line)
            CACHE_ROWS.append(validate_detection_row(row, index, video_hash, WIDTH, HEIGHT, CLASS_NAMES))
    require(len(CACHE_ROWS) == EXPECTED_FRAME_COUNT, 'Cache ended before all retained frames were available.')
    REPORT['input'] = {
        'video_path': str(VIDEO_PATH), 'video_sha256': video_hash, 'video_size_bytes': VIDEO_PATH.stat().st_size,
        'detection_report_path': str(DETECTION_REPORT_PATH), 'detection_report_sha256': sha256_file(DETECTION_REPORT_PATH),
        'frame_detections_path': str(FRAME_DETECTIONS_PATH), 'frame_detections_sha256': cache_hash,
        'parent_run_id': PARENT_RUN_ID, 'parent_status': PARENT_REPORT['status'],
        'expected_and_validated_rows': len(CACHE_ROWS), 'width': WIDTH, 'height': HEIGHT,
        'fps_from_parent_opencv': FPS, 'parent_input_metadata': PARENT_REPORT['input'],
        'parent_detector_configuration': PARENT_REPORT['configuration'],
        'parent_detector_environment': PARENT_REPORT['environment'],
        'parent_detector_weights': PARENT_REPORT['weights'],
        'timestamp_semantics': 'Tracking uses zero-based frame order; index/FPS is approximate, backend timestamps are unverified PTS.',
    }
    print('Validated retained cache:', FRAME_DETECTIONS_PATH, len(CACHE_ROWS), 'rows; no detection executed.')


In [ ]:
with stage('tracker_configuration'):
    YAML_PATH = package_root / 'cfg/trackers/botsort.yaml'
    PUBLISHED_DEFAULTS = YAML.load(YAML_PATH)
    TRACKER_CONFIG = dict(PUBLISHED_DEFAULTS, **TRACKER_OVERRIDES)
    require(TRACKER_CONFIG['tracker_type'] == 'botsort', 'Tracker substitution is not allowed.')
    require(TRACKER_CONFIG['with_reid'] is False and TRACKER_CONFIG['gmc_method'] == 'none',
            'This phase must not introduce ReID or GMC.')
    REPORT['tracker'] = {
        'class': 'ultralytics.trackers.bot_sort.BOTSORT', 'package_version': PACKAGE_VERSION,
        'published_yaml_sha256': sha256_file(YAML_PATH), 'published_defaults': PUBLISHED_DEFAULTS,
        'explicit_overrides': TRACKER_OVERRIDES, 'effective_configuration': TRACKER_CONFIG,
        'association_device': 'cpu', 'class_gate_in_verified_source': False,
        'class_semantics': 'Raw class/score follow the latest matched detection; no product mapping or stabilization.',
        'lost_buffer_semantics': 'Pinned constructor uses track_buffer directly as update-call/frame count, no FPS rescaling.',
        'max_frames_lost': TRACKER_CONFIG['track_buffer'],
        'buffer_seconds_estimate_from_parent_fps': TRACKER_CONFIG['track_buffer']/FPS,
        'internal_source_constants': {'second_stage_iou_cost_limit': 0.5, 'unconfirmed_cost_limit': 0.7,
                                      'tracked_lost_duplicate_iou_distance_ceiling': 0.15},
        'reid_architecture_status': 'Still target architecture; isolated disabled baseline precedes Phase 4.',
    }
    print(json.dumps(REPORT['tracker'], indent=2))


## Prototype evidence schema and lifecycle semantics

Every `track_records.jsonl` row corresponds to exactly one cached frame, including empty observations. `raw_detections` preserves parent evidence. `emitted_tracks` contains activated BoT-SORT outputs linked by the verified original detection index. A tracker box is Kalman-filtered; `raw_bbox` is the associated actual detector observation. Out-of-frame Kalman estimates are retained without clipping; rendering clips only drawing coordinates.

The observation-only subclass captures references to newly created tracks before pool deduplication. It does not alter association, thresholds, IDs or state transitions. Equality against uninstrumented BoT-SORT is checked on actual cached rows. Snapshots include confirmed/unconfirmed, lost, newly removed and disappeared-pool records. A `not_retained` state means the object disappeared from the library's retained pools without an explicit removal notification; it is not fabricated as `Removed`. State and pool membership remain separate because newly removed objects can remain in the lost pool for one update.

`track_summary.json` keeps all captured IDs, including never emitted/unconfirmed IDs. `trajectory` contains only ordered, detection-linked observations, including unconfirmed observations; it includes raw box centers as debug information and does not select the OPEN geometry anchor. `detection_count` counts these assignments once per frame; `missing_frame_count` is the consecutive frames since the most recent assignment at the ID's last recorded lifecycle snapshot. Gap events between assigned observations include the intervening missing frame indices. Lost/predicted snapshots are excluded from measured trajectory. `direction` and `counting_state` stay `None`.

ID totals, class changes, removals and gap events are software diagnostics. A gap/ID discontinuity/class change is an audit candidate, not automatically a confirmed physical-object ID switch or traffic event. No ground-truth metrics are produced.


In [ ]:
STATE_NAMES = {TrackState.New: 'new', TrackState.Tracked: 'tracked',
               TrackState.Lost: 'lost', TrackState.Removed: 'removed'}

class EvidenceBOTSORT(BOTSORT):
    """Observe library object references; preserve the parent association algorithm."""
    def update(self, results, img=None, feats=None, **kwargs):
        self.before_objects = list(self.tracked_stracks) + list(self.lost_stracks)
        self.created_objects = []
        return super().update(results, img=img, feats=feats, **kwargs)

    def _init_new_tracks(self, u_detection, detections, activated, refind=None):
        previous_length = len(activated)
        super()._init_new_tracks(u_detection, detections, activated, refind)
        self.created_objects.extend(activated[previous_length:])

def detection_boxes(row):
    matrix = np.asarray([det['bbox_xyxy_pixels'] + [det['confidence'], det['class_id']]
                         for det in row['detections']], dtype=np.float32).reshape(-1, 6)
    return Boxes(matrix, orig_shape=(row['height'], row['width']))

def validate_emitted(outputs, row):
    result = np.asarray(outputs, dtype=np.float32)
    if result.size == 0:
        result = np.empty((0, 8), dtype=np.float32)
    require(result.ndim == 2 and result.shape[1] == 8, 'Unexpected BoT-SORT output shape.')
    require(np.isfinite(result).all(), 'Nonfinite emitted tracker values.')
    ids, indices = set(), set()
    for output in result:
        x1, y1, x2, y2, track_id_float, score, cls_float, index_float = output.tolist()
        require(x2 > x1 and y2 > y1, 'Degenerate emitted Kalman box.')
        track_id, index, cls = int(track_id_float), int(index_float), int(cls_float)
        require(track_id_float == track_id and track_id > 0 and track_id not in ids, 'Invalid/duplicate emitted ID.')
        require(index_float == index and 0 <= index < len(row['detections']) and index not in indices,
                'Invalid/duplicate original detection linkage.')
        source = row['detections'][index]
        require(cls_float == cls and cls == source['class_id'], 'Emitted class is not latest linked raw class.')
        require(math.isclose(score, source['confidence'], abs_tol=1e-6), 'Emitted score provenance mismatch.')
        ids.add(track_id); indices.add(index)
    return result

def pool_signature(tracker):
    def signature(track):
        return (track.track_id, track.state, track.is_activated, track.frame_id, track.start_frame,
                int(track.idx), int(track.cls), float(track.score), tuple(track.xyxy.tolist()))
    return {name: [signature(track) for track in getattr(tracker, name, [])]
            for name in ['tracked_stracks', 'lost_stracks', 'removed_stracks', 'removed_stracks_frame']}

def frame_evidence(tracker, outputs, row, registry, previous_states):
    frame_index = row['frame_index']
    current_frame_id = frame_index + 1
    require(tracker.frame_id == current_frame_id, 'Tracker update count must equal cached frame index+1.')
    emitted_by_id = {int(out[4]): out for out in outputs}
    tracked = {t.track_id for t in tracker.tracked_stracks}
    lost = {t.track_id for t in tracker.lost_stracks}
    removed = {t.track_id for t in getattr(tracker, 'removed_stracks_frame', [])}
    objects = {t.track_id: t for t in tracker.before_objects + tracker.created_objects +
               tracker.tracked_stracks + tracker.lost_stracks + getattr(tracker, 'removed_stracks_frame', [])}
    snapshots, transitions, observations = [], [], []
    current_states = {}
    for track_id, track in sorted(objects.items()):
        membership = [name for name, pool in [('tracked', tracked), ('lost', lost), ('removed_this_frame', removed)]
                      if track_id in pool]
        retained = bool(membership)
        state = STATE_NAMES[track.state] if retained else 'not_retained'
        state_key = (state, bool(track.is_activated))
        current_states[track_id] = state_key
        entry = registry.setdefault(track_id, {
            'track_id': track_id, 'first_seen_frame': None, 'last_seen_frame': None,
            'detection_count': 0, 'emitted_observation_count': 0, 'missing_frame_count': 0,
            'class_id': None, 'class_name': None, 'confidence': None, 'bbox': None, 'center': None,
            'trajectory': [], 'gap_events': [], 'class_changes': [], 'lifecycle': [],
            'direction': None, 'counting_state': None,
        })
        observed = track.frame_id == current_frame_id
        if observed:
            detection_index = int(track.idx)
            require(0 <= detection_index < len(row['detections']), 'Lifecycle detection linkage out of range.')
            det = row['detections'][detection_index]
            require(int(track.cls) == det['class_id'] and math.isclose(float(track.score), det['confidence'], abs_tol=1e-6),
                    'Lifecycle class/score not backed by current raw detection.')
            require(entry['last_seen_frame'] is None or entry['last_seen_frame'] < frame_index,
                    'Per-ID history is not strictly ordered.')
            if entry['last_seen_frame'] is not None:
                gap = frame_index-entry['last_seen_frame']-1
                if gap:
                    entry['gap_events'].append({'previous_observed_frame': entry['last_seen_frame'],
                                                 'next_observed_frame': frame_index, 'missing_frames': gap})
                if entry['class_id'] != det['class_id']:
                    entry['class_changes'].append({'frame_index': frame_index, 'previous_class_id': entry['class_id'],
                                                   'new_class_id': det['class_id'], 'detection_index': detection_index})
            box = list(det['bbox_xyxy_pixels'])
            center = [(box[0]+box[2])/2, (box[1]+box[3])/2]
            observation = {
                'track_id': track_id, 'frame_index': frame_index, 'original_detection_index': detection_index,
                'raw_bbox': box, 'bbox': track.xyxy.tolist(), 'center': center,
                'class_id': det['class_id'], 'class_name': det['class_name'], 'confidence': det['confidence'],
                'timestamp_estimate_from_index_fps_seconds': frame_index/FPS,
                'timestamp_parent_backend_ms': row.get('timestamp_backend_ms'),
                'emitted': track_id in emitted_by_id, 'is_activated': bool(track.is_activated),
                'measurement_provenance': 'retained_phase_2_detection',
            }
            entry['trajectory'].append(observation)
            observations.append(observation)
            if entry['first_seen_frame'] is None: entry['first_seen_frame'] = frame_index
            entry['last_seen_frame'] = frame_index
            entry['detection_count'] += 1
            entry['emitted_observation_count'] += int(observation['emitted'])
            for field in ['class_id', 'class_name', 'confidence', 'bbox', 'center']:
                entry[field] = observation[field]
        entry['missing_frame_count'] = frame_index-entry['last_seen_frame'] if entry['last_seen_frame'] is not None else 0
        entry['last_lifecycle_frame'] = frame_index
        entry['final_recorded_state'] = state
        estimated_box = track.xyxy.tolist()
        require(all(math.isfinite(value) for value in estimated_box) and estimated_box[2] > estimated_box[0] and
                estimated_box[3] > estimated_box[1], 'Invalid finite/positive lifecycle Kalman estimate.')
        snapshot = {
            'track_id': track_id, 'state': state, 'library_state': STATE_NAMES[track.state],
            'pool_membership': membership, 'is_activated': bool(track.is_activated),
            'observed_this_frame': observed, 'emitted': track_id in emitted_by_id,
            'bbox_kalman_estimate': track.xyxy.tolist(),
            'bbox_provenance': 'filtered_current_detection' if observed else 'tracker_estimate_not_current_detection',
            'latest_observed_frame': track.frame_id-1, 'missing_frame_count': entry['missing_frame_count'],
        }
        snapshots.append(snapshot)
        old_state = previous_states.get(track_id)
        if old_state != state_key:
            event = {'track_id': track_id, 'frame_index': frame_index,
                     'previous_state': list(old_state) if old_state else None, 'state': state,
                     'is_activated': bool(track.is_activated), 'pool_membership': membership}
            transitions.append(event); entry['lifecycle'].append(event)
    for track_id in list(previous_states):
        if track_id not in current_states and previous_states[track_id][0] not in ('removed', 'not_retained'):
            event = {'track_id': track_id, 'frame_index': frame_index,
                     'previous_state': list(previous_states[track_id]), 'state': 'not_retained',
                     'is_activated': previous_states[track_id][1], 'pool_membership': []}
            transitions.append(event); registry[track_id]['lifecycle'].append(event)
            registry[track_id]['final_recorded_state'] = 'not_retained'
            previous_states[track_id] = ('not_retained', previous_states[track_id][1])
    previous_states.update(current_states)
    observed_map = {obs['track_id']: obs for obs in observations}
    require(set(emitted_by_id) <= set(observed_map), 'An emitted track lacks a current detector observation.')
    emitted = []
    for track_id, output in emitted_by_id.items():
        obs = dict(observed_map[track_id])
        obs['bbox'] = output[:4].tolist()
        emitted.append(obs)
    return {
        'video_sha256': EXPECTED_VIDEO_SHA256, 'parent_run_id': PARENT_RUN_ID, 'frame_index': frame_index,
        'width': row['width'], 'height': row['height'],
        'timestamp_estimate_from_index_fps_seconds': frame_index/FPS,
        'timestamp_parent_backend_ms': row.get('timestamp_backend_ms'),
        'raw_detections': row['detections'], 'emitted_tracks': emitted,
        'assigned_observations_including_unconfirmed': observations,
        'lifecycle_snapshots': snapshots, 'lifecycle_transitions': transitions,
    }


In [ ]:
with stage('cached_adapter_and_observer_validation'):
    baseline = BOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    observed_tracker = EvidenceBOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    registry, states = {}, {}
    for row in CACHE_ROWS[:200]:
        baseline_output = baseline.update(detection_boxes(row), img=None)
        observed_output = observed_tracker.update(detection_boxes(row), img=None)
        require(np.array_equal(baseline_output, observed_output), 'Evidence observer altered actual tracker output.')
        require(pool_signature(baseline) == pool_signature(observed_tracker), 'Evidence observer altered tracker lifecycle.')
        normalized = validate_emitted(observed_output, row)
        frame_evidence(observed_tracker, normalized, row, registry, states)
    empty = dict(CACHE_ROWS[0], frame_index=200, detections=[])
    result = observed_tracker.update(detection_boxes(empty), img=None)
    require(observed_tracker.frame_id == 201 and np.asarray(result).size == 0, 'Empty frame must age tracks without emitting observations.')
    fresh = EvidenceBOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    fresh_output = validate_emitted(fresh.update(detection_boxes(CACHE_ROWS[0]), img=None), CACHE_ROWS[0])
    require(len(fresh_output) and int(fresh_output[:, 4].min()) == 1, 'A new video tracker must begin a fresh ID sequence.')
    REPORT['adapter_validation'] = {
        'actual_cached_rows_checked': 200, 'output_and_lifecycle_equivalence': 'passed',
        'empty_update_aging': 'passed', 'fresh_id_reset': 'passed', 'detector_inference_executed': False,
        'scope': 'CPU cached adapter smoke before one full Phase 3 sequence; not a rerun of Phase 2.',
    }
    del baseline, observed_tracker, fresh, registry, states
    print(json.dumps(REPORT['adapter_validation'], indent=2))


In [ ]:
def color_for_id(track_id):
    return tuple(int(v) for v in np.random.default_rng(track_id).integers(80, 256, 3))

def render_tracks(frame, evidence):
    image = frame.copy()
    assigned = {obs['original_detection_index'] for obs in evidence['assigned_observations_including_unconfirmed']}
    for index, det in enumerate(evidence['raw_detections']):
        if index not in assigned:
            box = np.rint(det['bbox_xyxy_pixels']).astype(int)
            cv2.rectangle(image, tuple(box[:2]), tuple(box[2:]), (160, 160, 160), 1)
    for obs in evidence['assigned_observations_including_unconfirmed']:
        x1, y1, x2, y2 = np.rint(obs['bbox']).astype(int)
        color = color_for_id(obs['track_id']) if obs['emitted'] else (0, 150, 255)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2 if obs['emitted'] else 1)
        label = f"ID {obs['track_id']} {obs['class_name']} {obs['confidence']:.2f} d{obs['original_detection_index']}"
        if not obs['emitted']: label += ' UNCONF/NOT-EMITTED'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        tx = max(0, min(x1, image.shape[1]-tw-2)); ty = max(th+28, min(y1, image.shape[0]-2))
        cv2.rectangle(image, (tx, ty-th-4), (min(tx+tw+2, image.shape[1]-1), ty+3), (0, 0, 0), -1)
        cv2.putText(image, label, (tx+1, ty-1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    cv2.rectangle(image, (0, 0), (image.shape[1]-1, 25), (0, 0, 0), -1)
    cv2.putText(image, f"frame {evidence['frame_index']} | IDs {len(evidence['emitted_tracks'])} | cached BoT-SORT CPU, ReID/GMC off",
                (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return image

BURST_FRAMES = {start: [] for start, end in BURSTS}
TRACK_REGISTRY, PREVIOUS_STATES = {}, {}
TRACKER_TIMES, DECODE_TIMES = [], []
RENDER_SECONDS = 0.0
SELECTED_FRAME_ARTIFACTS = []
BURST_RECORDS = []

with stage('full_cached_tracking_and_aligned_video_decode'):
    tracker = EvidenceBOTSORT(IterableSimpleNamespace(**TRACKER_CONFIG))
    require(tracker.encoder is None and tracker.gmc.method is None, 'Appearance/GMC inference was unexpectedly activated.')
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    require(cap.isOpened(), 'Matching video decoder could not open.')
    require(int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) == WIDTH and int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) == HEIGHT,
            'Actual source dimensions do not match cache.')
    observed_cv_metadata = {'width': cap.get(cv2.CAP_PROP_FRAME_WIDTH), 'height': cap.get(cv2.CAP_PROP_FRAME_HEIGHT),
                            'fps': cap.get(cv2.CAP_PROP_FPS), 'frame_count': cap.get(cv2.CAP_PROP_FRAME_COUNT),
                            'backend': cap.getBackendName()}
    REPORT['actual_decode_metadata'] = observed_cv_metadata
    loop_started = time.perf_counter()
    try:
        with (RUN_DIR / 'track_records.jsonl').open('w') as output:
            for row in CACHE_ROWS:
                frame_index = row['frame_index']
                started = time.perf_counter()
                ok, frame = cap.read()
                DECODE_TIMES.append(time.perf_counter()-started)
                require(ok, f'Video decoder stopped before cached frame {frame_index}; EOF versus decode failure unknown.')
                require(frame.shape[:2] == (HEIGHT, WIDTH), 'Decoded dimensions changed relative to cache.')
                started = time.perf_counter()
                result = tracker.update(detection_boxes(row), img=None)
                TRACKER_TIMES.append(time.perf_counter()-started)
                emitted = validate_emitted(result, row)
                evidence = frame_evidence(tracker, emitted, row, TRACK_REGISTRY, PREVIOUS_STATES)
                evidence['tracker_update_wall_seconds'] = TRACKER_TIMES[-1]
                evidence['timestamp_current_decoder_backend_ms'] = float(cap.get(cv2.CAP_PROP_POS_MSEC))
                output.write(json.dumps(evidence, allow_nan=False)+'\n')
                matched_burst = next((start for start, end in BURSTS if start <= frame_index <= end), None)
                if matched_burst is not None or frame_index in REPRESENTATIVE_INDICES or frame_index in FOCUS_INDICES:
                    started = time.perf_counter()
                    annotated = render_tracks(frame, evidence)
                    if matched_burst is not None:
                        BURST_FRAMES[matched_burst].append((frame_index, annotated))
                        BURST_RECORDS.append(evidence)
                    if frame_index in REPRESENTATIVE_INDICES or frame_index in FOCUS_INDICES:
                        raw_path = RUN_DIR / 'samples' / f'frame_{frame_index:09d}_raw.png'
                        annotated_path = RUN_DIR / 'samples' / f'frame_{frame_index:09d}_tracked.png'
                        require(cv2.imwrite(str(raw_path), frame) and cv2.imwrite(str(annotated_path), annotated), 'PNG write failed.')
                        SELECTED_FRAME_ARTIFACTS.append({'frame_index': frame_index,
                            'raw': str(raw_path.relative_to(RUN_DIR)), 'raw_sha256': sha256_file(raw_path),
                            'annotated': str(annotated_path.relative_to(RUN_DIR)), 'annotated_sha256': sha256_file(annotated_path)})
                    RENDER_SECONDS += time.perf_counter()-started
                REPORT['frames_processed'] = frame_index+1
                if (frame_index+1) % 1000 == 0:
                    output.flush(); write_report()
                    print(f"Processed {frame_index+1}/{len(CACHE_ROWS)} cached frames; IDs are not vehicle totals.", flush=True)
            started = time.perf_counter()
            extra_ok, _ = cap.read()
            DECODE_TIMES.append(time.perf_counter()-started)
            require(not extra_ok, 'Video contains a decoded frame beyond the validated cache; alignment is incomplete.')
            REPORT['decode_stop'] = 'opencv_read_false_eof_or_decode_failure_unknown_after_expected_rows'
    finally:
        cap.release()
        REPORT['execution'] = {
            'frames_processed': REPORT['frames_processed'], 'loop_wall_seconds': time.perf_counter()-loop_started,
            'tracker_update_calls': len(TRACKER_TIMES), 'tracker_update_wall_seconds': sum(TRACKER_TIMES),
            'tracker_update_wall_ms_mean': statistics.mean(TRACKER_TIMES)*1000 if TRACKER_TIMES else None,
            'tracker_update_wall_ms_median': statistics.median(TRACKER_TIMES)*1000 if TRACKER_TIMES else None,
            'decoder_read_calls_including_final_probe': len(DECODE_TIMES), 'decode_read_wall_seconds': sum(DECODE_TIMES),
            'selected_frame_render_and_png_write_wall_seconds': RENDER_SECONDS,
            'timing_limits': 'Tracker counter includes Boxes adapter construction and actual update. Loop also includes lifecycle serialization/IO. No new detection runtime is measured.',
        }
        (RUN_DIR / 'track_summary.json').write_text(json.dumps({
            'video_sha256': EXPECTED_VIDEO_SHA256, 'run_id': RUN_ID, 'track_id_scope': 'one video tracker instance',
            'summary_semantics': 'All captured IDs including unconfirmed; no physical-object/traffic totals.',
            'tracks': list(TRACK_REGISTRY.values())}, indent=2, allow_nan=False)+'\n')
    require(REPORT['frames_processed'] == EXPECTED_FRAME_COUNT, 'Full cache was not consumed.')
    REPORT['status'] = 'completed_cached_sequence_manual_review_pending'
    print('Actual full cached tracking sequence completed:', REPORT['frames_processed'])


In [ ]:
def make_contact_sheet(frames, title, path, indices=None, crop=None):
    if indices is not None:
        frames = [frames[index] for index in indices]
    columns = 2
    tile_width = 640
    tile_height = 360 if crop is None else round(tile_width*(crop[3]-crop[1])/(crop[2]-crop[0]))
    rows = math.ceil(len(frames)/columns)
    sheet = Image.new('RGB', (columns*tile_width, 40+rows*(tile_height+24)), 'black')
    draw = ImageDraw.Draw(sheet)
    draw.text((10, 10), title, fill='white')
    for position, (frame_index, bgr) in enumerate(frames):
        if crop is not None:
            x1,y1,x2,y2 = crop; bgr = bgr[y1:y2, x1:x2]
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tile = Image.fromarray(rgb).resize((tile_width, tile_height), Image.Resampling.LANCZOS)
        x = (position%columns)*tile_width; y = 40+(position//columns)*(tile_height+24)
        draw.text((x+8, y+4), f'frame {frame_index}', fill='white')
        sheet.paste(tile, (x, y+24))
    sheet.save(path)

with stage('temporal_evidence_export_and_consistency_audit'):
    temporal_artifacts = []
    for start, end in BURSTS:
        frames = BURST_FRAMES[start]
        require([index for index, _ in frames] == list(range(start, end+1)), 'Temporal burst is incomplete/out of order.')
        clip_path = RUN_DIR / 'temporal' / f'burst_{start:09d}_{end:09d}_tracked.mp4'
        writer = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*'mp4v'), FPS, (WIDTH, HEIGHT))
        require(writer.isOpened(), 'Temporal MP4 writer unavailable; no silent codec substitution.')
        try:
            for _, image in frames: writer.write(image)
        finally:
            writer.release()
        require(clip_path.is_file() and clip_path.stat().st_size > 0, 'Temporal clip was not retained.')
        probe = cv2.VideoCapture(str(clip_path)); decoded_frames = 0
        try:
            while True:
                ok, _ = probe.read()
                if not ok: break
                decoded_frames += 1
        finally:
            probe.release()
        require(decoded_frames == len(frames), 'Saved temporal clip does not decode every burst frame.')
        sheet_path = RUN_DIR / 'temporal' / f'burst_{start:09d}_{end:09d}_contact_sheet.png'
        make_contact_sheet(frames, f'BoT-SORT full-history temporal evidence {start}-{end}; display only', sheet_path,
                           indices=[0, 5, 10, 15, 20, 29])
        temporal_artifacts.append({'start_frame': start, 'end_frame': end, 'frame_count': decoded_frames,
            'clip': str(clip_path.relative_to(RUN_DIR)), 'clip_sha256': sha256_file(clip_path),
            'contact_sheet': str(sheet_path.relative_to(RUN_DIR)), 'contact_sheet_sha256': sha256_file(sheet_path)})
    focus_frames = [(index, image) for index, image in BURST_FRAMES[7696] if index in FOCUS_INDICES]
    require([index for index, _ in focus_frames] == FOCUS_INDICES, 'Focused scooter sequence incomplete.')
    focus_path = RUN_DIR / 'temporal' / 'entering_scooter_000007711_000007718_contact_sheet.png'
    make_contact_sheet(focus_frames, 'Entering scooter frames 7711-7718; display crop only; actual retained tracker IDs',
                       focus_path, crop=FOCUS_CROP_XYXY)
    (RUN_DIR / 'representative_temporal_track_records.json').write_text(json.dumps(BURST_RECORDS, indent=2, allow_nan=False)+'\n')
    rows_checked = 0
    emitted_observations = 0
    empty_detection_frames = 0
    no_emitted_frames = 0
    observed_track_ids = set()
    with (RUN_DIR / 'track_records.jsonl').open() as handle:
        for index, line in enumerate(handle):
            record = json.loads(line)
            require(record['frame_index'] == index and record['video_sha256'] == EXPECTED_VIDEO_SHA256, 'Serialized frame identity mismatch.')
            require(record['raw_detections'] == CACHE_ROWS[index]['detections'], 'Serialized raw detection evidence changed.')
            ids = [obs['track_id'] for obs in record['emitted_tracks']]
            require(len(ids) == len(set(ids)), 'Serialized duplicate emitted IDs.')
            observed_track_ids.update(ids)
            emitted_observations += len(ids)
            empty_detection_frames += int(not record['raw_detections'])
            no_emitted_frames += int(not ids)
            rows_checked += 1
    require(rows_checked == EXPECTED_FRAME_COUNT, 'Serialized tracker row total differs from cache.')
    for entry in TRACK_REGISTRY.values():
        history = entry['trajectory']
        require(entry['detection_count'] == len(history), 'Track detection_count differs from observed trajectory.')
        indices = [observation['frame_index'] for observation in history]
        require(indices == sorted(set(indices)), 'Track observation history is not strictly ordered.')
        require(entry['direction'] is None and entry['counting_state'] is None, 'Later-phase state was introduced.')
    REPORT['diagnostics'] = {
        'serialized_rows_verified': rows_checked, 'captured_tracker_ids_including_unconfirmed': len(TRACK_REGISTRY),
        'unique_emitted_tracker_ids': len(observed_track_ids), 'emitted_track_observations': emitted_observations,
        'frames_with_zero_cached_detections': empty_detection_frames, 'frames_with_no_emitted_tracks': no_emitted_frames,
        'ids_never_emitted': sum(entry['emitted_observation_count'] == 0 for entry in TRACK_REGISTRY.values()),
        'within_id_gap_events': sum(len(entry['gap_events']) for entry in TRACK_REGISTRY.values()),
        'within_id_raw_class_changes': sum(len(entry['class_changes']) for entry in TRACK_REGISTRY.values()),
        'lifecycle_transition_events': sum(len(entry['lifecycle']) for entry in TRACK_REGISTRY.values()),
        'id_switch_count': None, 'id_switch_count_status': 'Unknown; requires physical-object temporal review, not inferred from diagnostics.',
        'traffic_count': None, 'traffic_count_status': 'Not implemented; ID totals are not vehicle counts.',
    }
    REPORT['consistency_audit'] = {'status': 'passed', 'frame_order_and_video_cache_alignment': 'passed',
        'raw_detection_preservation': 'passed', 'emitted_id_and_detection_linkage': 'passed',
        'ordered_track_histories': 'passed', 'saved_temporal_clip_decoding': 'passed'}
    REPORT['artifacts'] = {'track_records': 'track_records.jsonl',
        'track_records_sha256': sha256_file(RUN_DIR/'track_records.jsonl'),
        'track_summary': 'track_summary.json', 'track_summary_sha256': sha256_file(RUN_DIR/'track_summary.json'),
        'representative_temporal_track_records': 'representative_temporal_track_records.json',
        'sample_frames': SELECTED_FRAME_ARTIFACTS, 'temporal_bursts': temporal_artifacts,
        'focused_scooter_contact_sheet': str(focus_path.relative_to(RUN_DIR)),
        'focused_scooter_contact_sheet_sha256': sha256_file(focus_path)}
    REPORT['limitations'] = [
        'No ground truth or physical-object identity labels; no model/tracking/counting quality metric is available.',
        'Only retained YOLO26M detections are replayed; detector confidence filtering already removed lower-scored/missing boxes.',
        'Motion/IoU/score association is class-agnostic in this pinned implementation; raw class changes are retained, not mapped to product categories.',
        'ReID and GMC are explicitly disabled for this baseline. Native detector appearance features were not retained in Phase 2.',
        'Five short temporal bursts and the focused sequence do not establish identity correctness throughout the complete video.',
        'Frame order is reproducible; backend times and index/FPS estimates do not establish source PTS, CFR/VFR or source decode integrity.',
        'No source stabilization, night enhancement, geometry eligibility, crossing or traffic counting is implemented.',
        'Observer reports not-retained pool disappearance separately from explicit library Removed state.',
        'Loop time includes decode, CPU association, bookkeeping, serialization and selected sample writes; it is not an end-to-end detector/tracker benchmark.',
    ]
    REPORT['finished_utc'] = datetime.now(timezone.utc).isoformat()
    write_report()
    print(json.dumps(REPORT['diagnostics'], indent=2))
    print('Evidence retained:', RUN_DIR)
write_report()
(RUN_DIR/'run_manifest.json').write_text(json.dumps(REPORT, indent=2, allow_nan=False)+'\n')


## Review after execution

Inspect the saved full-history temporal clips/contact sheets and corresponding native frames before recording findings. Review continuity, temporary loss/reactivation, newly assigned IDs, class changes and cropped/overlaid objects. `within_id_gap_events` and other diagnostics identify candidates only. A visual observation of the same physical vehicle continuing under a different ID needs an evidence-linked temporal note; ambiguous cases stay OPEN.

The Kaggle execution report remains `manual_review_pending` until the actual evidence is reviewed. Preserve that original export and store a separate AI-assisted/manual review supplement with its provenance. Do not overwrite pending fields to fabricate an in-notebook review. Phase 4 is a later authorized task, with actual appearance capability/weights verified before any comparison.
